# Day 26: Month 2 Project Kickoff

From today until Day 33 we build **one project**, one piece every day.

## 🎯 The project we will build in class

A **support bot for Chai Wala**, our tea shop in Gulberg, Lahore. A customer asks a question. The bot answers with the **real** shop facts.

| Customer asks | Bot answers |
|---|---|
| *How much is Kashmiri chai?* | Kashmiri chai is Rs 200 for one cup. |
| *Do you deliver to DHA?* | Yes. Delivery is Rs 100, and free if your order is Rs 1500 or more. |
| *Do you sell pizza?* | Sorry, we do not sell pizza. Would you like chai and a samosa? |

**Your own project:** you build the **same steps** for your own idea (see the Home Problem).

## 🗺️ The project plan (Day 26 → Day 33)

```
Day 33   Test and polish             friends test your bot
  ▲
Day 32   Better answers              cite sources, say "I don't know"
  ▲
Day 31   Search + answer (RAG)       the bot finds facts, then answers
  ▲
Day 30   Documents → chunks          cut the shop documents into small parts
  ▲
Day 29   Embeddings + Chroma         store the facts as numbers
  ▲
Day 28   How LLMs work               what happens inside the API
  ▲
Day 27   Chatbot + web page          memory and a Streamlit app
  ▲
Day 26   One good prompt             ← TODAY
```

## 📋 Agenda

```
TOP     Step 6   Put it together          the full Day 26 bot
          ▲
        Step 5   Test it                  5 questions, check each answer
          ▲
        Step 4   Write a good prompt      5 parts
          ▲
        Step 3   A weak prompt            see the bot make up facts
          ▲
        Step 2   The .env file            keep the key safe
          ▲
BOTTOM  Step 1   Choose a use case        what will your bot do?
```

### New words today

| Word | Meaning |
|---|---|
| **Use case** | The real job the bot does, for real people. |
| **Prototype** | A first, simple version that works. We make it better later. |
| **.env file** | A small text file on your laptop that keeps secrets, like the API key. |
| **Hallucination** | When the model **makes up** a fact that is not true. |
| **System prompt** | The rules for the model, sent at the top of every chat (Day 24). |

---
# Class Problem: the Chai Wala support bot

## Step 1: Choose a use case

A good use case has **3 things**:

1. **Real users**: who will use it? (Chai Wala customers)
2. **Real questions**: what do they ask again and again? (prices, delivery, timings)
3. **Facts you have**: where do the answers come from? (the shop's menu and rules)

Pick **one** of these 3 for your own project:

| Use case | Users | Example question |
|---|---|---|
| Customer support bot | Customers of a shop | *Do you deliver to DHA?* |
| Study assistant | Students of one course | *What is a list in Python?* |
| Business FAQ bot | Visitors of a business | *What are your timings?* |

**✏️ Quick Check:** For the **Chai Wala** bot, write the 3 things: users, 3 real questions, and where the facts come from.

In [ ]:
# your code here

**Answer:** Users: Chai Wala customers. Questions: *How much is chai? Do you deliver to Model Town? Are you open on Friday?* Facts: the shop menu, the delivery rules and the timings.

## Step 2: The .env file

On Day 24 we pasted the key every time. Now we save it **once** in a `.env` file.

1. In the **main folder** of the course, make a file with the name `.env`
2. Write one line in it (your own key): `OPEN_ROUTER_KEY=sk-or-...`
3. Check that `.gitignore` has `.env` in it. Then the key never goes to GitHub.
4. Install the reader once: `pip install python-dotenv`

`load_dotenv()` reads the file. If there is no `.env` file, `getpass` asks for the key.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

In [ ]:
def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

print(ask([{"role": "user", "content": "Say hello in 5 words."}]))

## Step 3: A weak prompt

We give the model only a name and no facts. Then we ask a real customer question.

In [ ]:
weak = [{"role": "system", "content": "You are the support bot of Chai Wala, a tea shop in Lahore."},
        {"role": "user",   "content": "How much is the delivery charge to DHA?"}]

print(ask(weak))

**What went wrong?** The model does not know our shop. It **makes up** a price (a **hallucination**), or it gives a long, unclear answer.

The real answer is: **Rs 100, free if the order is Rs 1500 or more.** The model cannot know this. **We must give it the facts.**

## Step 4: Write a good prompt

A good system prompt has **5 parts**:

| Part | What it says | Chai Wala |
|---|---|---|
| 1. **Role** | Who is the bot? | You are the support bot of Chai Wala. |
| 2. **Facts** | What does it know? | Menu, prices, delivery, timings, payment. |
| 3. **Rules** | What must it do? | Use only these facts. Prices in Rs. |
| 4. **Style** | How does it talk? | Friendly, short: max 3 sentences. |
| 5. **Unknown** | What if the answer is not in the facts? | Say sorry and give the phone number. |

*Everyday example:* a new worker at the shop on day 1. You tell him who he is, give him the price list, the rules, how to talk to customers, and who to call if he does not know.

In [ ]:
FACTS = """
Menu: doodh patti chai Rs 120, Kashmiri chai Rs 200, green tea Rs 100,
plain paratha Rs 80, aloo paratha Rs 150, samosa Rs 60, cake rusk pack Rs 250.
Delivery: Gulberg, DHA, Model Town, Johar Town. Charge Rs 100, free for orders of Rs 1500 or more.
Delivery takes 30 to 45 minutes. Last order at 10:30 pm.
Timings: every day 9 am to 11 pm. Friday closed 1 pm to 2 pm for Jumma.
Payment: cash, card, JazzCash, Easypaisa. Cash on delivery is available.
Phone: 042-1234567. WhatsApp: 0300-1234567.
"""

SYSTEM_PROMPT = f"""
You are the support bot of Chai Wala, a tea shop in Gulberg, Lahore.

Facts:
{FACTS}

Rules:
- Use ONLY the facts above. Never make up a price, place or time.
- Write prices like "Rs 120".
- Be friendly. Reply in at most 3 short sentences.
- If the answer is not in the facts, say: "Sorry, I don't know. Please call 042-1234567."
"""

In [ ]:
def support_bot(question):
    return ask([{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question}])

print(support_bot("How much is the delivery charge to DHA?"))

**✏️ Quick Check:** Ask the bot: *"I want to pay with a cheque. Is that ok?"* Cheques are **not** in the facts. What should a good bot say?

In [ ]:
# your code here

**Answer:** It should **not** say yes. Cheques are not in the facts, so it should say *"Sorry, I don't know. Please call 042-1234567."* (or say that the payment options are cash, card, JazzCash and Easypaisa).

In [ ]:
print(support_bot("I want to pay with a cheque. Is that ok?"))

## Step 5: Test it

We never trust **one** answer. We write a small **test list**: questions and the answer we **expect**. Then we read every reply.

In [ ]:
tests = [
    ("How much is Kashmiri chai?",          "Rs 200"),
    ("Do you deliver to Johar Town?",       "yes, Rs 100"),
    ("Are you open on Friday at 1:30 pm?",  "no, Jumma break"),
    ("Do you sell pizza?",                  "no / I don't know"),
    ("What is the capital of Pakistan?",    "I don't know, call the shop"),
]

for question, expected in tests:
    print("Q:", question)
    print("Expected:", expected)
    print("Bot:", support_bot(question))
    print()

**Read each answer.** Is the fact right? Is it short? If one is wrong, change the **rules** in the prompt (not the question), and run the test list again.

**One problem for later:** all the facts are inside the prompt. A big shop has 100 pages of facts. That is too many tokens for every question (Day 24: tokens = cost). From **Day 29** the bot will **search** for only the facts it needs.

**✏️ Quick Check:** Add one more test: *"I ordered 10 cups of doodh patti chai. Is delivery free?"* What is the right answer?

In [ ]:
# your code here

**Answer:** 10 × Rs 120 = Rs 1200. That is less than Rs 1500, so delivery is **not** free: it is Rs 100. (Models are sometimes wrong with numbers. Check it!)

In [ ]:
print(support_bot("I ordered 10 cups of doodh patti chai. Is delivery free?"))

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

FACTS = """
Menu: doodh patti chai Rs 120, Kashmiri chai Rs 200, green tea Rs 100,
plain paratha Rs 80, aloo paratha Rs 150, samosa Rs 60, cake rusk pack Rs 250.
Delivery: Gulberg, DHA, Model Town, Johar Town. Charge Rs 100, free for orders of Rs 1500 or more.
Delivery takes 30 to 45 minutes. Last order at 10:30 pm.
Timings: every day 9 am to 11 pm. Friday closed 1 pm to 2 pm for Jumma.
Payment: cash, card, JazzCash, Easypaisa. Cash on delivery is available.
Phone: 042-1234567. WhatsApp: 0300-1234567.
"""

SYSTEM_PROMPT = f"""
You are the support bot of Chai Wala, a tea shop in Gulberg, Lahore.

Facts:
{FACTS}

Rules:
- Use ONLY the facts above. Never make up a price, place or time.
- Write prices like "Rs 120".
- Be friendly. Reply in at most 3 short sentences.
- If the answer is not in the facts, say: "Sorry, I don't know. Please call 042-1234567."
"""

def support_bot(question):
    return ask([{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question}])

for q in ["How much is Kashmiri chai?", "Do you deliver to Johar Town?", "Do you sell pizza?"]:
    print("Q:", q, "\nBot:", support_bot(q), "\n")

### Key takeaways

- A project starts with a **use case**: real users, real questions, facts you have.
- Keep the key in a **.env** file. `.gitignore` keeps it off GitHub.
- With no facts, the model **makes up** answers (hallucination).
- A good system prompt has **5 parts**: role, facts, rules, style, unknown.
- Always **test** with a list of questions and expected answers.
- Facts inside the prompt cost tokens. From Day 29 we **search** for the facts.

---
# 🏠 Home Problem

**Part A: Chai Wala has news.** Change the prompt and test it.

- New item: **Chicken roll, Rs 250.**
- New rule: **On rainy days, delivery takes up to one hour.**
- New rule: **The bot must reply in Roman Urdu if the customer writes in Roman Urdu.**

1. Add the new facts to `FACTS` and the new rule to `SYSTEM_PROMPT`.
2. Add 3 tests to the test list: one for each change. Example: *"Chicken roll kitne ka hai?"*
3. Run all 8 tests. Write one line for each: right or wrong?

**Part B: your project.** Choose your own use case (Step 1 table). Write the 3 things (users, 3 questions, facts) and a 5-part system prompt for it. Bring it to Day 27.